---
#### 6-2. Model tunning 
- Grid Search, Ramdomized Search 
---

In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

#### make_classification (학습용 인공데이터)

In [2]:
# 데이터 생성 및 검증용 분리 
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=2000,
    n_features=20,
    n_informative=5,      # 실제 중요한 변수 적게
    n_redundant=5,
    n_repeated=0,
    n_classes=2,
    flip_y=0.12,           # 노이즈 추가
    class_sep=0.6,        # 클래스 겹치게 (난이도 ↑)
    random_state=42
)

print(X[:1])
print(y[:10])

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_valid.shape, y_train.shape, y_valid.shape)

[[ 0.04788878  1.44862483 -0.28168832 -0.44438084  2.12168455  1.13116789
   0.36548229  0.55294319  0.80976035 -0.98001645 -1.46288716  0.37050201
  -1.56207026 -1.40125401 -0.64324843 -0.27483238 -0.81686681 -1.59043026
  -0.60913609 -0.76608046]]
[1 1 0 0 0 1 1 0 1 1]
(1400, 20) (600, 20) (1400,) (600,)


In [3]:
# Baseline 1
rf1 = RandomForestClassifier(
    n_estimators=10,
    max_depth=2,
    min_samples_leaf=15,
    random_state=42
)

rf1.fit(X_train, y_train)
pred1 = rf1.predict(X_valid)

acc1 = accuracy_score(y_valid, pred1)
print("RF1 Accuracy:", round(acc1, 4))

RF1 Accuracy: 0.7033


In [4]:
# Baseline 2
rf2 = RandomForestClassifier(
    n_estimators=80,
    max_depth=6,
    min_samples_leaf=5,
    random_state=42
)

rf2.fit(X_train, y_train)
pred2 = rf2.predict(X_valid)

acc2 = accuracy_score(y_valid, pred2)
print("RF2 Accuracy:", round(acc2, 4))

RF2 Accuracy: 0.79


In [5]:
# Grid Search
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train, y_train)

best_rf_grid = grid.best_estimator_
pred_grid = best_rf_grid.predict(X_valid)

acc_grid = accuracy_score(y_valid, pred_grid)

print("GridSearch Accuracy:", round(acc_grid, 4))
print("Best Params:", grid.best_params_)

GridSearch Accuracy: 0.8383
Best Params: {'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 300}


In [6]:
# Random Search 
from scipy.stats import randint

param_dist = {
    'n_estimators': randint(100, 600),
    'max_depth': randint(5, 30),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10)
}

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=30,
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

best_rf_random = random_search.best_estimator_
pred_random = best_rf_random.predict(X_valid)

acc_random = accuracy_score(y_valid, pred_random)

print("RandomSearch Accuracy:", round(acc_random, 4))
print("Best Params:", random_search.best_params_)

RandomSearch Accuracy: 0.8367
Best Params: {'max_depth': 25, 'min_samples_leaf': 1, 'min_samples_split': 13, 'n_estimators': 413}


#### 위스콘신 유방암 데이터 

In [7]:
# data loading 
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer()
X = data.data
y = data.target

print('features :', data.feature_names, '\n')
print('target :', data.target_names, '\n')

print(X[:1], '\n')
print(y[:10])

features : ['mean radius' 'mean texture' 'mean perimeter' 'mean area'
 'mean smoothness' 'mean compactness' 'mean concavity'
 'mean concave points' 'mean symmetry' 'mean fractal dimension'
 'radius error' 'texture error' 'perimeter error' 'area error'
 'smoothness error' 'compactness error' 'concavity error'
 'concave points error' 'symmetry error' 'fractal dimension error'
 'worst radius' 'worst texture' 'worst perimeter' 'worst area'
 'worst smoothness' 'worst compactness' 'worst concavity'
 'worst concave points' 'worst symmetry' 'worst fractal dimension'] 

target : ['malignant' 'benign'] 

[[1.799e+01 1.038e+01 1.228e+02 1.001e+03 1.184e-01 2.776e-01 3.001e-01
  1.471e-01 2.419e-01 7.871e-02 1.095e+00 9.053e-01 8.589e+00 1.534e+02
  6.399e-03 4.904e-02 5.373e-02 1.587e-02 3.003e-02 6.193e-03 2.538e+01
  1.733e+01 1.846e+02 2.019e+03 1.622e-01 6.656e-01 7.119e-01 2.654e-01
  4.601e-01 1.189e-01]] 

[0 0 0 0 0 0 0 0 0 0]


In [8]:
# 검증데이터 분리 (훈련난이도 상승 위해 40%로 설정)
X_train, X_valid, y_train, y_valid = train_test_split( 
    X, y, test_size=0.4, stratify=y, random_state=42 
) 

print(X_train.shape, X_valid.shape, y_train.shape, y_valid.shape)

(341, 30) (228, 30) (341,) (228,)


In [9]:
# 1-1. Base model 1
rf = RandomForestClassifier(
    n_estimators=10,
    max_depth=2,
    min_samples_leaf=10,
    random_state=42
)

rf.fit(X_train, y_train)

pred = rf.predict(X_valid)
acc = accuracy_score(y_valid, pred)
f1 = f1_score(y_valid, pred)

print("============= Base Line Model 1 ==================================")
print("ACC of Base1 =", round(acc, 4))
print("F1 of Base1 =", round(f1, 4))

============= Base Line Model 1 ==================================
ACC of Base1 = 0.9298
F1 of Base1 = 0.9437


In [10]:
# 1-2. Base model 2
rf2 = RandomForestClassifier(
    n_estimators=50,
    max_depth=5,
    min_samples_leaf=3,
    random_state=42
)

rf2.fit(X_train, y_train)

pred2 = rf2.predict(X_valid)
acc = accuracy_score(y_valid, pred2)
f1 = f1_score(y_valid, pred2)

print("============= Base Line Model 2 ==================================")
print("ACC of Base2 =", round(acc, 4))
print("F1 of Base2 =", round(f1, 4))

============= Base Line Model 2 ==================================
ACC of Base2 = 0.943
F1 of Base2 = 0.9547


In [11]:
# 2. Grid Search 
param_grid = {
      'n_estimators': [100, 200, 300, 400],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train, y_train)

best_rf = grid.best_estimator_
pred_tuned = best_rf.predict(X_valid)

acc = accuracy_score(y_valid, pred_tuned)
f1 = f1_score(y_valid, pred_tuned)

print("============= Grid Search Model ===============================")
print("ACC =", round(acc, 4))
print("F1  =", round(f1, 4))
print("Best Params:", grid.best_params_)
print("Best CV Score:", round(grid.best_score_, 4))   # cv 5회의 f1 score 평균 

============= Grid Search Model ===============================
ACC = 0.9474
F1  = 0.958
Best Params: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Best CV Score: 0.9677


In [12]:
# 3. Random Search 
param_dist = {
    'n_estimators': randint(100, 600),
    'max_depth': randint(3, 30),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10)
}

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=20,          # 랜덤 샘플링 횟수
    cv=5,
    scoring='f1',
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

best_rf_random = random_search.best_estimator_
pred_random = best_rf_random.predict(X_valid)

acc_random = accuracy_score(y_valid, pred_random)
f1_random = f1_score(y_valid, pred_random)

print("============= Random Search Model ===============================")
print("ACC =", round(acc_random, 4))
print("F1  =", round(f1_random, 4))
print("Best Params:", random_search.best_params_)
print("Best CV Score:", round(random_search.best_score_, 4))

============= Random Search Model ===============================
ACC = 0.9474
F1  = 0.958
Best Params: {'max_depth': 6, 'min_samples_leaf': 2, 'min_samples_split': 9, 'n_estimators': 487}
Best CV Score: 0.9676


- Breast Cancer 데이터는 양이 적고 복잡성도 크게 높지 않은 편이라, 기본모델에서 약간의 튜닝만으로 성능 한계 도달 